In [2]:
# -*- coding: utf-8 -*-
"""3D FDM diagnostics for the apparent x-refinement error plateau.

The existing 901--2401 sequence changes x and time together while keeping the
periodic directions fixed.  These cases separate temporal and transverse-grid
effects before any claim is made about an "equivalent" uniform grid.
"""

from __future__ import annotations

import math
import os
from typing import Dict, List, Tuple

import pandas as pd
import torch

from hv_cnlf_common_verified import (
    CaseOptions,
    load_fixed_lhs,
    run_imex_cnlf_case,
    save_case_outputs,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
BASE_PATH = os.environ.get("BENCHMARK_BASE_PATH", ".")
OUTPUT_DIR = os.environ.get(
    "BENCHMARK_OUTPUT_DIR",
    os.path.join(BASE_PATH, "verified_fdm_3d_diagnostics_outputs"),
)

X_MIN, X_MAX = -1.0, 1.0
Y_MIN, Y_MAX = -1.0, 1.0
Z_MIN, Z_MAX = -1.0, 1.0
T_FINAL = 0.5
LEFT_BC, RIGHT_BC = -4.0, 2.0

MU_LIST = [1.0e-2]
DAE_TARGET_E2 = {1.0e-2: 7.43e-3}
NUM_SAMPLES = 13000
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# Direct DAE-LHS extraction requires Nx-1, Ny-1, Nz-1, and Nt-1 to be
# divisible by 50.  Update DAE_TARGET_E2 from the final DAE table before run.
# CASES: Dict[float, List[Tuple[int, int, int, int]]] = {
#     1.0e-2: [
#         (2401, 451, 451, 11251),
#         # Reproduce the unpublished 3201-grid table entry with a runtime JSON.
#         (3201, 401, 401, 15001),
#     ],
# }
CASES: Dict[float, List[Tuple[int, int, int, int]]] = {
    1.0e-2: [
        # 现有的基准
        (2401, 451, 451, 11251),
        (3201, 401, 401, 15001),
        
        # --- 新增的 10 组加密参数 ---
        # (3501, 451, 451, 16001),
        # (3201, 501, 501, 15001),
        # (3501, 501, 501, 17501),
        # (4001, 501, 501, 18501),
        # (4001, 551, 551, 20001),
        # (4501, 551, 551, 22501),
        # (4501, 601, 601, 25001),
        # (5001, 601, 601, 25001),
        # (5001, 701, 701, 28001),
        # (6001, 801, 801, 30001), 
    ],
}
# Run this expensive combined refinement only if the diagnostics show that
# transverse resolution materially lowers e2 and the H200 has enough memory.
if os.environ.get("RUN_3D_COMBINED", "0") == "1":
    CASES[1.0e-2].append((3201, 451, 451, 15001))

def find_reference_file(mu: float) -> str:
    mu_id = int(round(-math.log10(mu)))
    candidates = [
        f"3d_U0_true_mu{mu:.0e}.csv",
        f"3d_U0_all_t_u_x_y_z_t_mu{mu_id}_51_mathematica_619.csv",
    ]
    if abs(mu - 1.0e-2) < 1.0e-15:
        candidates.append("3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv")
    for name in candidates:
        if os.path.exists(os.path.join(BASE_PATH, name)):
            return name
    raise FileNotFoundError("Cannot find 3D reference file. Tried: " + ", ".join(candidates))


def analytical_integral_minus(
    x: torch.Tensor, y: torch.Tensor, z: torch.Tensor
) -> torch.Tensor:
    a = y - x
    b = z - x
    term1 = 6.0 * torch.sin(torch.pi * x) * torch.cos(torch.pi * (a - b))
    term2 = 3.0 * torch.sin(torch.pi * (x + a + b))
    term3 = torch.sin(torch.pi * (3.0 * x + a + b))
    term4 = 4.0 * torch.sin(torch.pi * (a + b))
    return (term1 + term2 + term3 + term4) / (12.0 * torch.pi)


def phi_minus(x: torch.Tensor, y: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    integral = analytical_integral_minus(x, y, z)
    return -torch.sqrt(torch.clamp(16.0 + 2.0 * integral, min=1.0e-10))


def phi_plus(x: torch.Tensor, y: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    integral = analytical_integral_minus(x, y, z)
    return torch.sqrt(torch.clamp(4.0 + 2.0 * integral, min=1.0e-10))


def asymptotic_initial_condition(
    x: torch.Tensor, y: torch.Tensor, z: torch.Tensor, mu: float
) -> torch.Tensor:
    pm_xyz, pp_xyz = phi_minus(x, y, z), phi_plus(x, y, z)
    zero_x = torch.zeros_like(y + z)
    pm_h, pp_h = phi_minus(zero_x, y, z), phi_plus(zero_x, y, z)
    jump = pp_h - pm_h

    arg_m = torch.clamp((-x * jump) / (2.0 * mu), -500.0, 500.0)
    arg_p = torch.clamp((x * jump) / (2.0 * mu), -500.0, 500.0)
    u_m = pm_xyz + jump / (torch.exp(arg_m) + 1.0)
    u_p = pp_xyz - jump / (torch.exp(arg_p) + 1.0)
    return torch.where(x <= 0.0, u_m, u_p)


def source_f(
    x: torch.Tensor, y: torch.Tensor, z: torch.Tensor
) -> torch.Tensor:
    return (
        torch.cos(torch.pi * x)
        * torch.cos(torch.pi * y)
        * torch.cos(torch.pi * z)
    )


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("=" * 112)
    print("3D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark")
    print(f"device={DEVICE}, dtype={DTYPE}")
    print("x: Dirichlet; y,z: periodic; FFT fast Helmholtz solver")
    print("T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded")
    print("Fixed DAE LHS set; direct grid-node extraction; no interpolation")
    print("=" * 112)

    rows = []
    for mu in MU_LIST:
        reference = find_reference_file(mu)
        lhs = load_fixed_lhs(
            reference_path=os.path.join(BASE_PATH, reference),
            coordinate_columns=("t", "x", "y", "z"),
            index_path=os.path.join(BASE_PATH, f"3d_LHS_sample_indices_mu{mu:.0e}_51_13000.npy"),
            num_samples=NUM_SAMPLES,
            save_points_path=os.path.join(OUTPUT_DIR, f"3d_HV_LHS_points_mu{mu:.0e}_51_13000.csv"),
        )

        for nx, ny, nz, nt in CASES[mu]:
            print("-" * 112)
            print(f"mu={mu:.0e}, grid={nx}x{ny}x{nz}, Nt={nt}")
            options = CaseOptions(
                dim=3,
                mu=mu,
                grid=(nx, ny, nz),
                nt=nt,
                bounds=((X_MIN, X_MAX), (Y_MIN, Y_MAX), (Z_MIN, Z_MAX)),
                t_final=T_FINAL,
                left_bc=LEFT_BC,
                right_bc=RIGHT_BC,
                dtype=DTYPE,
                device=DEVICE,
                eval_warmup=EVAL_WARMUP,
                eval_repeat=EVAL_REPEAT,
                finite_check_interval=max(1, (nt - 1) // 100),
                progress_every=max(1, (nt - 1) // 20),
                cfl_warning=1.0,
            )
            try:
                result = run_imex_cnlf_case(
                    options=options,
                    lhs_data=lhs,
                    source_function=source_f,
                    initial_function=asymptotic_initial_condition,
                )
                result["target_e2"] = float(DAE_TARGET_E2[mu])
                result["pass_target"] = bool(
                    float(result["e2"]) <= float(DAE_TARGET_E2[mu])
                )
                prefix = os.path.join(
                    OUTPUT_DIR,
                    f"3d_HV_IMEX_CNLF_diagnostic_mu{mu:.0e}_Nx{nx}_Ny{ny}_Nz{nz}_Nt{nt}",
                )
                pred_path, runtime_path = save_case_outputs(
                    result, lhs, ("t", "x", "y", "z"), prefix
                )
                row = {k: v for k, v in result.items() if k != "prediction"}
                row.update({
                    "status": "success", "failure_reason": "",
                    "Nx": nx, "Ny": ny, "Nz": nz,
                    "prediction_csv": pred_path,
                    "runtime_json": runtime_path,
                })
                print(
                    f"e2={result['e2']:.6e}, einf={result['einf']:.6e}, "
                    f"T_solve={result['T_solve']:.6f}s, "
                    f"T_eval={result['T_eval']:.6e}s, "
                    f"T_total={result['T_total']:.6f}s, "
                    "peak GPU allocated="
                    f"{result['peak_gpu_allocated_gib']:.3f} GiB"
                )
            except (RuntimeError, MemoryError) as exc:
                row = {
                    "status": "failed", "failure_reason": str(exc),
                    "mu": mu, "Nx": nx, "Ny": ny, "Nz": nz, "Nt": nt,
                    "target_e2": float(DAE_TARGET_E2[mu]),
                    "pass_target": False,
                }
                print(f"FAILED: {exc}")
            rows.append(row)
            pd.DataFrame(rows).to_csv(
                os.path.join(OUTPUT_DIR, "3d_HV_IMEX_CNLF_diagnostics_summary.csv"), index=False
            )
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    summary = os.path.join(OUTPUT_DIR, "3d_HV_IMEX_CNLF_diagnostics_summary.csv")
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(summary, index=False)
    selected_path = os.path.join(OUTPUT_DIR, "3d_HV_IMEX_CNLF_diagnostics_selected.csv")
    if not summary_df.empty and "pass_target" in summary_df.columns:
        passed = summary_df[
            (summary_df["status"] == "success")
            & summary_df["pass_target"].fillna(False).astype(bool)
        ].copy()
        if not passed.empty:
            sort_columns = [c for c in ("mu", "spatial_unknowns", "Nt") if c in passed.columns]
            passed = passed.sort_values(sort_columns)
            selected = passed.groupby("mu", as_index=False).first()
            selected.to_csv(selected_path, index=False)
        else:
            pd.DataFrame(columns=summary_df.columns).to_csv(selected_path, index=False)
    print(f"Saved summary: {summary}")
    print(f"Saved selected passing cases: {selected_path}")


if __name__ == "__main__":
    main()


3D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark
device=cuda, dtype=torch.float64
x: Dirichlet; y,z: periodic; FFT fast Helmholtz solver
T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded
Fixed DAE LHS set; direct grid-node extraction; no interpolation
Loaded fixed DAE LHS set: N_test=13000, reference=3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv, indices=3d_LHS_sample_indices_mu1e-02_51_13000.npy
----------------------------------------------------------------------------------------------------------------
mu=1e-02, grid=2401x451x451, Nt=11251
step=1/11250, t=4.444444e-05, max|u|=4.068852e+00
step=562/11250, t=2.497778e-02, max|u|=4.068183e+00
step=1124/11250, t=4.995556e-02, max|u|=4.067558e+00
step=1686/11250, t=7.493333e-02, max|u|=4.067039e+00
step=2248/11250, t=9.991111e-02, max|u|=4.066677e+00
step=2810/11250, t=1.248889e-01, max|u|=4.066512e+00
step=3372/11250, t=1.498667e-01, max|u|=4.066494e+00
step=3934/11250, t=1.748444e-01, max|u|=4.066493e+00
step=449